# Ejercicio 2 — Optimización Multivariable

**Objetivo:** minimizar f(x, y) = x² + y².

**Codificación:** cromosoma de 8 bits, dividido a la mitad → los primeros 4 bits codifican `x` y los últimos 4 bits codifican `y`. Cada mitad se decodifica al rango `[-5, 5]`.

**Aptitud:** como el AG está diseñado para *maximizar*, y queremos *minimizar* f(x, y), usamos como aptitud el valor negativo: `aptitud = -f(x, y)`. Así, maximizar la aptitud equivale a minimizar la función original.

## 0. Importaciones y semilla aleatoria

In [ ]:
import random
import matplotlib.pyplot as plt

random.seed(42)


## 1. Núcleo genérico del Algoritmo Genético

Codificación binaria, selección por torneo, cruce de un punto y mutación bit a bit.

In [ ]:
def crear_individuo(n_bits):
    """Genera un cromosoma binario aleatorio de longitud n_bits."""
    return [random.randint(0, 1) for _ in range(n_bits)]


def crear_poblacion(tam_poblacion, n_bits):
    """Genera la población inicial de individuos binarios."""
    return [crear_individuo(n_bits) for _ in range(tam_poblacion)]


def bin_a_decimal(bits):
    """Convierte una lista de bits (0/1) a su valor decimal entero."""
    valor = 0
    for bit in bits:
        valor = valor * 2 + bit
    return valor


def decodificar_rango(bits, x_min, x_max):
    """Mapea un segmento binario al rango real [x_min, x_max]."""
    n_bits = len(bits)
    valor_decimal = bin_a_decimal(bits)
    max_decimal = 2 ** n_bits - 1
    return x_min + (valor_decimal / max_decimal) * (x_max - x_min)


In [ ]:
def seleccion_torneo(poblacion, aptitudes, k=3):
    """Selecciona un padre mediante torneo de tamaño k."""
    seleccionados = random.sample(range(len(poblacion)), k)
    mejor = max(seleccionados, key=lambda i: aptitudes[i])
    return poblacion[mejor]


def cruce(padre1, padre2, pc=0.8):
    """Cruce de un punto entre dos padres, con probabilidad pc."""
    if random.random() < pc:
        punto = random.randint(1, len(padre1) - 1)
        hijo1 = padre1[:punto] + padre2[punto:]
        hijo2 = padre2[:punto] + padre1[punto:]
        return hijo1, hijo2
    return padre1[:], padre2[:]


def mutacion(individuo, pm):
    """Invierte cada bit con probabilidad pm (mutación bit a bit)."""
    return [bit if random.random() > pm else 1 - bit for bit in individuo]


def elitismo(poblacion, aptitudes, n_elite=1):
    """Preserva intactos a los n_elite mejores individuos de la generación actual."""
    indices_ordenados = sorted(range(len(poblacion)), key=lambda i: aptitudes[i], reverse=True)
    mejores_indices = indices_ordenados[:n_elite]
    return [poblacion[i][:] for i in mejores_indices]


In [ ]:
def ejecutar_ag(fitness_func, n_bits, tam_poblacion=40, generaciones=60,
                 pm=0.1, pc=0.8, n_elite=1):
    """
    Ejecuta el algoritmo genético completo.
    Devuelve: mejor individuo encontrado, su aptitud, e historial de convergencia.
    """
    poblacion = crear_poblacion(tam_poblacion, n_bits)
    historial_mejor = []
    mejor_individuo_global = None
    mejor_aptitud_global = float('-inf')

    for gen in range(generaciones):
        aptitudes = [fitness_func(ind) for ind in poblacion]

        idx_mejor_gen = max(range(len(poblacion)), key=lambda i: aptitudes[i])
        if aptitudes[idx_mejor_gen] > mejor_aptitud_global:
            mejor_aptitud_global = aptitudes[idx_mejor_gen]
            mejor_individuo_global = poblacion[idx_mejor_gen][:]
        historial_mejor.append(mejor_aptitud_global)

        nueva_poblacion = elitismo(poblacion, aptitudes, n_elite)

        while len(nueva_poblacion) < tam_poblacion:
            padre1 = seleccion_torneo(poblacion, aptitudes)
            padre2 = seleccion_torneo(poblacion, aptitudes)
            hijo1, hijo2 = cruce(padre1, padre2, pc)
            hijo1 = mutacion(hijo1, pm)
            hijo2 = mutacion(hijo2, pm)
            nueva_poblacion.append(hijo1)
            if len(nueva_poblacion) < tam_poblacion:
                nueva_poblacion.append(hijo2)

        poblacion = nueva_poblacion

    return mejor_individuo_global, mejor_aptitud_global, historial_mejor


## 2. Codificación y aptitud del Ejercicio 2

In [ ]:
X_MIN, X_MAX = -5, 5
Y_MIN, Y_MAX = -5, 5


def decodificar_ej2(bits):
    """Divide el cromosoma en dos mitades y decodifica x e y por separado."""
    mitad = len(bits) // 2
    bits_x = bits[:mitad]
    bits_y = bits[mitad:]
    x = decodificar_rango(bits_x, X_MIN, X_MAX)
    y = decodificar_rango(bits_y, Y_MIN, Y_MAX)
    return x, y


def fitness_ej2(bits):
    """Aptitud = -f(x, y), para convertir la minimización en maximización."""
    x, y = decodificar_ej2(bits)
    f = x**2 + y**2
    return -f


## 3. Ejecutar el algoritmo genético

In [ ]:
mejor_bits_2, mejor_apt_2, hist_2 = ejecutar_ag(fitness_ej2, n_bits=8, n_elite=1)
mejor_x, mejor_y = decodificar_ej2(mejor_bits_2)

print(f"Mejor x = {mejor_x:.4f}")
print(f"Mejor y = {mejor_y:.4f}")
print(f"f(x, y) minimizado = {-mejor_apt_2:.6f}")


## 4. Gráfica de convergencia

In [ ]:
plt.figure()
plt.plot([-v for v in hist_2])
plt.title("Convergencia Ejercicio 2 - Minimización f(x, y) = x² + y²")
plt.xlabel("Generación")
plt.ylabel("Mejor valor de f(x, y) encontrado")
plt.grid(True)
plt.show()


## 5. Conclusión

El valor teórico óptimo es `f(0, 0) = 0`. El AG se acerca a ese valor, pero la precisión está limitada por la resolución de la codificación binaria (con 4 bits por variable solo existen 16 puntos posibles en el rango [-5, 5]). Aumentar el número de bits por variable mejoraría la precisión a costa de un espacio de búsqueda más grande.